# Lunar Lander

This environment is part of the Box2D environments which contains general information about the environment.
https://gymnasium.farama.org/environments/box2d/lunar_lander/

### Description
This environment is a classic rocket trajectory optimization problem. According to Pontryagin’s maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off.

There are two environment versions: discrete or continuous. The landing pad is always at coordinates (0,0). The coordinates are the first two numbers in the state vector. Landing outside of the landing pad is possible. Fuel is infinite, so an agent can learn to fly and then land on its first attempt.

### Action Space
There are four discrete actions available:

0: do nothing

1: fire left orientation engine

2: fire main engine

3: fire right orientation engine

### Observation Space
The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

### Rewards
After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

is increased/decreased the closer/further the lander is to the landing pad.

is increased/decreased the slower/faster the lander is moving.

is decreased the more the lander is tilted (angle not horizontal).

is increased by 10 points for each leg that is in contact with the ground.

is decreased by 0.03 points each frame a side engine is firing.

is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.

Should be running in conda environment 'd:\installe_software\lunarlander'

In [60]:
# %pip install gymnasium

In [61]:
# pip install swig
"""
The pip install swig does not work in windows. Creates problem with gymnasium[box2d] installation.
I had to install swig manually:
1. Download the 'swigwin-4.4.0' from https://www.swig.org/download.html
2. Unzip it to a folder swigwin-4.4.0'
3. Add the path to the swigwin-4.4.0 folder to the system environment variable 'Path'
4. Restart VS Code
5. Now "ModuleNotFoundError: No module named 'swig'" this should be resolved.
6. Check by running 'swig -version' in the terminal.

"Microsoft Visual C++ 14.0 or greater is required" if you face this error 
while installing gymnasium[box2d], then:
1. "vs_BuildTools" from "https://visualstudio.microsoft.com/visual-cpp-build-tools/"
2. While installing select "Desktop development with C++"
"""

'\nThe pip install swig does not work in windows. Creates problem with gymnasium[box2d] installation.\nI had to install swig manually:\n1. Download the \'swigwin-4.4.0\' from https://www.swig.org/download.html\n2. Unzip it to a folder swigwin-4.4.0\'\n3. Add the path to the swigwin-4.4.0 folder to the system environment variable \'Path\'\n4. Restart VS Code\n5. Now "ModuleNotFoundError: No module named \'swig\'" this should be resolved.\n6. Check by running \'swig -version\' in the terminal.\n\n"Microsoft Visual C++ 14.0 or greater is required" if you face this error \nwhile installing gymnasium[box2d], then:\n1. "vs_BuildTools" from "https://visualstudio.microsoft.com/visual-cpp-build-tools/"\n2. While installing select "Desktop development with C++"\n'

In [62]:
# %pip install gymnasium[box2d]

# Dummy environment without any RL

In [4]:
import gymnasium as gym

# Create the LunarLander-v3 environment with custom parameters
env = gym.make("LunarLander-v3", continuous=False, gravity=-10.0,
               enable_wind=True, wind_power=15.0, turbulence_power=1.5,
               render_mode="human") # render_mode enables visualization

# Reset environment
env.reset()
print(env.action_space)

done = False
total_reward = 0.0
repeat = 3

for episode in range(repeat):
    observation, info = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        # Taking random action (Can be replaced with agent's action)
        action = env.action_space.sample()

        # Step in environment
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        # Check if episode finished
        done = terminated or truncated

    print(f"Episode finished total reward: {total_reward}")

env.close()

Discrete(4)
Episode finished total reward: -100.03621753850071
Episode finished total reward: -565.0444398234304
Episode finished total reward: -122.90096837830117


In [ ]:
import random
import numpy as np 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import gymnasium as gym


In [65]:
# Setting up the environment
env = gym.make("LunarLander-v3", continuous=False, gravity=-10.0,
               enable_wind=False, wind_power=15.0, turbulence_power=1.5)    # render_mode="human"  : Slows down training

In [66]:
state = env.observation_space.shape
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print(f"States: {state} \nState size: {state_size} \nAction size: {action_size}")

States: (8,) 
State size: 8 
Action size: 4


In [67]:
learning_rate = 5e-4
minibatch = 150
gamma = 0.99
replay_buffer_size = 100000
interpolation_parameter = 1e-3
number_episodes = 5000
max_time_steps = 1000
epsilon_starting_value = 1.0
epsilon_ending_value = 0.01
epsilon_decay_value = 0.995
scores_100_episode = deque(maxlen=100)

### The Aritficial Neural Network is used to decide what action the agent should take at a given state

In [68]:
class ANN(nn.Module):
    def __init__(self, state_size, action_size, seed = 42):
        super(ANN, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = self.fc1(state)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

### Replay Memory : 
#### This class is designed to manage the agent's memory of the game experiences. It stores the state, action, reward, next-state and wheather the game ended for each step in the game.

experience = (state, action, reward, next-state, done)

In [69]:
class ReplayMemory(object):
    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []

    # Adding new event to the replay memory
    def push(self, event):
        if len(self.memory) > self.capacity:
            del self.memory[0]  # Remove the oldest event from memory
        self.memory.append(event)

    # Taken rendom samples from the memory of size batch_size in the form of torch tensors
    def sample(self, batch_size):
        experiences = random.sample(self.memory, batch_size)
        # Converting values to torch tensors
        states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float()
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long()  # gather() funciton needs a long(int64) type tensor
        rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float()
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float()
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float()

        return states, actions, rewards, next_states, dones

### Local Network(Policy Network) vs Target Network
local Q-network and target Q-network is a design choice in Deep Q-Learning(DQN) to improve the stablity and convergence of training

Local Q-Network(self.local_network):
1. Actively update duing training.
2. Used to predict Q-values for the current state when the agent selects actions

Target Q-Network (self.target_network):
1. Used to compute the target Q-value for the next during training.
2. Updated less frequently than the local Q-network to provide stable targets.

In [70]:
class Agent():
    def __init__(self, state_size, action_size, seed = 42):
        self.state_size = state_size
        self.action_size = action_size
        self.local_qnetwork = ANN(state_size, action_size)
        self.target_qnetwork = ANN(state_size, action_size)
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(),
                                    lr = learning_rate)
        self.memory = ReplayMemory(replay_buffer_size)
        self.t_step = 0     # Training step

    # The step function will be called by the agent after each action
    # Agent will pass the experience (experience = state, action, reward, next_state, done)
    # Than we will store the experience in the memeory
    def step(self, state, action, reward, next_state, done):
        self.memory.push((state, action, reward, next_state, done)) # Pass the experienc as a tuple
        self.t_step = (self.t_step + 1) % 4 # update every 4 time steps
        if self.t_step == 0:
            if len(self.memory.memory) > minibatch:
                experiences = self.memory.sample(minibatch)
                self.learn(experiences, gamma)

    # This method decides which action the agent should take based on the current state
    # It also handles the exploration vs exploitation trade-off using epsilon-greedy policy
    def get_action(self, state, epsilon):
        # Convert the state from a numpy array to a torch tensor
        state = torch.from_numpy(state).float().unsqueeze(0)
        # Putting the local network to evaluation mode to avoid unwanted updates
        self.local_qnetwork.eval()
        with torch.no_grad():
            action_values = self.local_qnetwork(state) # Get action values from the local Q-network
        
        # Putting the local network back to training mode
        self.local_qnetwork.train()
        # Epsilon-greedy action selection
        if random.random() > epsilon:
            # We will pick the value with the highest action value
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    # The learn method is used to train an AI agent in Reinforcement Learning
    def learn(self, experiences, gamma):
        # Unpacking the experiences
        states, actions, rewards, next_states, dones = experiences
        
        # We pass the batch of next_states throught the target Q-network to get Q-values 
        # for all possible actions :- self.target_qnetwork(next_states)
        # Shape : (batch_size, num_actions)
        # batch_size is the number of experience samples in the minibatch
        # num_actions is the number of actions the agent can take
        # The target network is used to calculate the target Q-values and we do not want to update
        # its weights during backpropagation, se we use .detach()
        # detach() removes the computaion graph from the tensor, stopping gradients from 
        # flowing through it. 
        # So during backpropagation we only update the local Q-network not the target Q-network
        # max(1)[0] finds the maximum Q-value for each next_state across all possible actions.
        next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)

        # Calculating the target Q-values using the Bellman equation
        # if it is terminal state, dones = 1
        # so we take only the gamma. 
        q_targets = rewards + (gamma * next_q_targets * (1 - dones))

        # Calculating the expected Q-values for the local network
        # Use the 'actions' index tensor to extract the Q-value that was predicted 
        # for the specific action actually taken for each state in the batch.
        q_expected = self.local_qnetwork(states).gather(1, actions)

        # Loss calculation
        # Here we are using Mean Squared Error(MSE) between 
        # predicted Q-values - 'q_expected' and target Q-values - 'q_targets'
        loss = F.mse_loss(q_expected, q_targets)

        # Reset the gradient to zero from the previous step to prevent accumulation
        self.optimizer.zero_grad()

        # Backpropagation: Compute gradients of the loss with respect to model parameters
        loss.backward()  

        # Updata the model parameters using the computed gradients to minimize the loss
        self.optimizer.step()  # Update the weights

        # Updating the target Q-network using soft update method\
        self.soft_update(self.local_qnetwork, self.target_qnetwork, interpolation_parameter)


    def soft_update(self, local_qnetwork, target_qnetwork, interpolation_parameter):
        for target_params, local_params in zip(target_qnetwork.parameters(), local_qnetwork.parameters()):
            target_params.data.copy_(interpolation_parameter * local_params.data + 
                                     (1.0 - interpolation_parameter) * target_params.data)


In [71]:
# Initializing the agent
agent = Agent(state_size, action_size)

In [ ]:
epsilon = epsilon_starting_value
for episode in range(0, number_episodes):
    state, _ = env.reset()
    score = 0
    for ts in range(0, max_time_steps):
        action = agent.get_action(state, epsilon)
        next_state, reward, done, _, _ =env.step(action)
        agent.step(state, action, reward, next_state, done)
        state = next_state  # Now next state becomes the current state
        score += reward
        if done:
            break

    # Storing score for each episode
    scores_100_episode.append(score)

    # Updating epsilon for exploration-exploitation trade-off
    epsilon = max(epsilon_ending_value, epsilon * epsilon_decay_value)

    if episode % 10 == 0:
         print(f"Episode: {episode} | Avg Score: {np.mean(scores_100_episode):.2f}")
    if np.mean(scores_100_episode) >= 205:  # Winig score is actually 200
        print(f"Congratulations! Environment solved in {episode} episodes!")
        torch.save(agent.local_qnetwork.state_dict(), 'lunar_lander_model.pth')
        break

Episode: 0 | Avg Score: -449.60
Episode: 10 | Avg Score: -183.12
Episode: 20 | Avg Score: -170.59
Episode: 30 | Avg Score: -176.73
Episode: 40 | Avg Score: -178.30
Episode: 50 | Avg Score: -163.12
Episode: 60 | Avg Score: -163.90
Episode: 70 | Avg Score: -167.35
Episode: 80 | Avg Score: -164.42
Episode: 90 | Avg Score: -165.05
Episode: 100 | Avg Score: -158.55
Episode: 110 | Avg Score: -156.89
Episode: 120 | Avg Score: -154.70
Episode: 130 | Avg Score: -149.93
Episode: 140 | Avg Score: -143.75
Episode: 150 | Avg Score: -148.45
Episode: 160 | Avg Score: -142.12
Episode: 170 | Avg Score: -132.27
Episode: 180 | Avg Score: -126.65
Episode: 190 | Avg Score: -113.08
Episode: 200 | Avg Score: -107.15
Episode: 210 | Avg Score: -95.88
Episode: 220 | Avg Score: -87.02
Episode: 230 | Avg Score: -73.72
Episode: 240 | Avg Score: -64.93
Episode: 250 | Avg Score: -60.16
Episode: 260 | Avg Score: -53.43
Episode: 270 | Avg Score: -48.26
Episode: 280 | Avg Score: -43.32
Episode: 290 | Avg Score: -46.18


In [76]:
import base64
import imageio
import gymnasium as gym
from IPython.display import HTML, display

def record_agent_video(agent, env_name, output_filename="trained_agent.mp4", fps=30):
    """Record a video of the trained agent playing the lunarlander-v3 discrete environment"""
    env = gym.make(env_name, continuous=False, render_mode="rgb_array")
    state, _ = env.reset()
    terminated = False
    truncated = False
    frames = []

    while not (terminated or truncated):
        frames.append(env.render())
        action = agent.get_action(state, 0.0) # Epsilon value is 0.0 as the model is already trained
        state, _, terminated, truncated, _ = env.step(action.item())

    env.close()
    imageio.mimsave(output_filename, frames, fps=fps)

def display_video(filename = "trained_agent.mp4"):
    """Display recorded video inside a jupiter notebook"""
    try:
        with open(filename, "rb") as video_file:
            encoded_video = base64.b64encode(video_file.read()).decode("ascii")
        display(HTML(f"""
                     <video alt="Agent Playing" autoplay loop controls style="height:400px;">
                        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4";
                     </video>
                     """))
    except FileNotFoundError:
        print("Error: Vido file not found")

record_agent_video(agent, "LunarLander-v3")
display_video()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
